# NAME TAG V2 Agent Smoke Test
Mock LLM을 사용해 Agent의 simple/adaptive/complex 실행을 재현합니다.
이 노트북은 외부 API를 호출하지 않으며, 실행이 끝나면 임시 SQLite DB를 정리합니다.

In [17]:
import os
import sys
from collections.abc import Generator

from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '../backend')
load_dotenv(Path("../backend/.env"), override=True)
os.environ['LLM_PROVIDER'] = 'gemini'

from fastapi.testclient import TestClient
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, sessionmaker
from sqlalchemy.pool import StaticPool

from app.main import app
from app.db.base import Base
from app.dependencies import get_database
from app.llm.gateway import get_llm_gateway
from app.config import get_settings
from app.models.agent_run import AgentRun
from app.models.tool_call import ToolCall
from app.models.brand_state import BrandState
from app.models.history import History
from app.models.snapshot import Snapshot

get_settings.cache_clear()
get_llm_gateway.cache_clear()
engine = create_engine(
    'sqlite://',
    connect_args={'check_same_thread': False},
    poolclass=StaticPool,
)
Base.metadata.create_all(engine)
SessionFactory = sessionmaker(bind=engine, autoflush=False, autocommit=False)

def override_get_database() -> Generator[Session, None, None]:
    db = SessionFactory()
    try:
        yield db
    finally:
        db.close()

app.dependency_overrides[get_database] = override_get_database
client = TestClient(app)
print('Mock LLM and isolated SQLite database are ready.')

Mock LLM and isolated SQLite database are ready.


In [18]:
signup = client.post('/api/auth/signup', json={
    'email': 'agent-notebook@example.com',
    'password': 'Password123!',
})
assert signup.status_code == 201, signup.text
brand_response = client.post('/api/brands', json={
    'name': 'Notebook Brand',
    'description': 'Agent smoke test brand',
})
assert brand_response.status_code == 201, brand_response.text
brand_id = brand_response.json()['id']
print({'brand_id': brand_id})

{'brand_id': 'c81a9022-e543-4ca8-9b21-3074fee47d6c'}


## 1. Simple request
BrandState를 조회하고 실행 결과를 반환하는 기본 Agent Loop입니다.

In [19]:
simple = client.post('/api/agent/chat', json={
    'brand_id': brand_id,
    'message': '현재 브랜드 상태를 보여줘',
})
assert simple.status_code == 200, simple.text
simple_result = simple.json()
print({
    'message': simple_result['message'],
    'agent': simple_result['agent'],
    'execution': simple_result['execution'],
    'selected_skill': simple_result['context'].get('selected_skill'),
})

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'message': '현재 등록된 브랜드 정보가 없습니다. NAME TAG와 함께 비즈니스 아이디어를 구체화하여 브랜드를 구축해 보세요. 먼저 어떤 산업 분야의 사업을 구상 중이신가요?', 'agent': {'run_id': '8752dd5f-4997-41b7-a572-1cc51ee1c602', 'mode': 'adaptive', 'status': 'completed'}, 'execution': {'mode': 'adaptive', 'status': 'completed', 'iterations': 3, 'tools_used': ['get_brand_state', 'get_brand_context']}, 'selected_skill': 'brand'}


## 2. Complex positioning request
Research Job 관찰, Employee Meeting, Positioning Skill, Proposal 생성을 확인합니다.

In [20]:
complex = client.post('/api/agent/chat', json={
    'brand_id': brand_id,
    'message': '시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘',
})
assert complex.status_code == 200, complex.text
complex_result = complex.json()
assert complex_result['agent']['mode'] == 'complex'
assert complex_result['context']['selected_skill'] == 'generate_positioning'
print({
    'agent': complex_result['agent'],
    'execution': complex_result['execution'],
    'proposal_id': complex_result['proposal_id'],
    'research': complex_result['context']['tool_results'].get('get_research_job'),
})

{'agent': {'run_id': '4d4f47f4-fbdf-4b46-94c2-877ceb0d4c55', 'mode': 'complex', 'status': 'completed'}, 'execution': {'mode': 'complex', 'status': 'completed', 'iterations': 5, 'tools_used': ['get_brand_state', 'get_brand_context', 'create_research_job', 'get_research_job', 'run_employee_meeting']}, 'proposal_id': 'c2e46e7a-f339-4eae-b01c-651ddaa5fe6d', 'research': {'job_id': '6cffab94-08e6-4120-a1a9-d6517098c263', 'brand_id': 'c81a9022-e543-4ca8-9b21-3074fee47d6c', 'mode': 'deep', 'status': 'planning', 'query': '시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘', 'plan': {'title': '시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘', 'objective': '시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘', 'questions': ['시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘'], 'scope': ['시장', '고객', '경쟁사'], 'output_type': 'market_analysis'}, 'progress': {'stage': 'planning', 'sources_found': 0}}}


In [21]:
db = SessionFactory()
try:
    runs = db.scalars(select(AgentRun).where(AgentRun.brand_id == brand_id).order_by(AgentRun.created_at.asc())).all()
    print('AgentRuns:')
    for run in runs:
        calls = db.scalars(select(ToolCall).where(ToolCall.agent_run_id == run.id).order_by(ToolCall.created_at.asc())).all()
        print({
            'run_id': run.id,
            'route': run.route,
            'status': run.status,
            'mode': run.result.get('mode'),
            'tools': [call.tool_name for call in calls],
            'tool_statuses': [call.status for call in calls],
        })

    state_before = db.scalar(select(BrandState).where(BrandState.brand_id == brand_id))
    print({'state_version_before_approval': state_before.version, 'positioning_before_approval': state_before.state['brand']['positioning']})
    assert state_before.version == 1
    assert state_before.state['brand']['positioning'] is None
finally:
    db.close()

AgentRuns:
{'run_id': '8752dd5f-4997-41b7-a572-1cc51ee1c602', 'route': 'brand', 'status': 'completed', 'mode': 'adaptive', 'tools': ['get_brand_state', 'get_brand_context'], 'tool_statuses': ['completed', 'completed']}
{'run_id': '4d4f47f4-fbdf-4b46-94c2-877ceb0d4c55', 'route': 'brand', 'status': 'completed', 'mode': 'complex', 'tools': ['get_brand_state', 'get_brand_context', 'create_research_job', 'get_research_job', 'run_employee_meeting'], 'tool_statuses': ['completed', 'completed', 'completed', 'completed', 'completed']}
{'state_version_before_approval': 1, 'positioning_before_approval': None}


## 3. Approval, Snapshot, History
Proposal을 승인하고 적용한 뒤 BrandState version, Snapshot, History를 확인합니다.

In [22]:
proposal_id = complex_result['proposal_id']
assert proposal_id
approved = client.post(f'/api/brands/{brand_id}/proposals/{proposal_id}/approve')
assert approved.status_code == 200, approved.text
applied = client.post(f'/api/brands/{brand_id}/proposals/{proposal_id}/apply')
assert applied.status_code == 200, applied.text

db = SessionFactory()
try:
    state_after = db.scalar(select(BrandState).where(BrandState.brand_id == brand_id))
    snapshots = db.scalars(select(Snapshot).where(Snapshot.brand_id == brand_id)).all()
    histories = db.scalars(select(History).where(History.brand_id == brand_id)).all()
    print({
        'state_version_after_apply': state_after.version,
        'positioning_after_apply': state_after.state['brand']['positioning'],
        'snapshot_count': len(snapshots),
        'history_count': len(histories),
    })
    assert state_after.version == 2
    assert snapshots
    assert histories
finally:
    db.close()

{'state_version_after_apply': 2, 'positioning_after_apply': '시장과 경쟁사 근거를 조사해서 포지셔닝을 전면적으로 개선해줘', 'snapshot_count': 1, 'history_count': 1}


## Cleanup
노트북을 다시 실행하기 전에 기존 TestClient override를 제거합니다.

In [7]:
app.dependency_overrides.clear()
client.close()
Base.metadata.drop_all(engine)
engine.dispose()
get_settings.cache_clear()
get_llm_gateway.cache_clear()
print('Agent smoke-test resources cleaned up.')

Agent smoke-test resources cleaned up.
